# Pass 1: Grief & Outrage + Context
LLM coding of grief/outrage in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Sentiment labels (POSITIVE / NEGATIVE / NEUTRAL) quantify tone but collapse
moral emotions. Pass 1 isolates Grief and Outrage specifically, plus the
value/relationship invoked as grounds and whose family is referenced —
the moral core of crisis discourse.

### Approach
Chunked LLM calls over slim `df[["video_id", "text"]]` CSV pasted into the
versioned prompt in `../prompts/pass-01-grief-outrage.md`. Output strictly
JSON: `grief_or_outrage ⊆ {Grief, Outrage}`, `value_or_relationship_invoked`,
`family_referenced[]`. Empty array / "None" when absent.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1. Load** | Read micro-sample | Preprocessed CSV → `n_sample=30` rows |
| **2. Preprocess** | Slim + clean | Keep `video_id, text`; `dropna`; `strip` |
| **3. Chunk** | Split rows | `chunk_size=15` → `chunk.to_csv(index=False)` per call |
| **4. Call** | LLM coding | `generate_content`, strip fences, `json.loads`, retry-once |
| **5. Export** | Save + inspect | `sample_pass01_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *Which videos concentrate outrage vs grief?*
- *What values/relationships ground each emotion?*
- *Whose family is invoked, and where?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass01_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import random

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 200
chunk_size = 5
sample_seed = 42
model = "gemini-3.8-flash"
overwrite = False

In [ ]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
OUTPUT_PATH = f"../output/{crisis_name}/sample_pass01_results.jsonl"
LOG_PATH    = f"../output/{crisis_name}/sample_pass01_failures.log"
PARTIAL_RESP = OUTPUT_PATH + ".partial.jsonl"
PARTIAL_FAIL = LOG_PATH + ".partial.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-4-31b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 8192  # max output ceiling; model uses only what it needs
REQUEST_TIMEOUT_MS = 60000  # per-request cap; stalls raise into transient retry
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)

## Micro-sample ingest & preprocess
Load `n_sample` rows from `INPUT_PATH`, slim to `video_id` + `text`,
drop nulls, strip whitespace. Keeps `row_id` for lossless merge at inspect time.

In [ ]:
full_df = pd.read_csv(INPUT_PATH)
print(f"Full phase rows: {len(full_df)}")

sample_df = full_df.head(n_sample).copy()  # deterministic for review; n_sample=200
sample_df = sample_df[["video_id", "text"]]
sample_df = sample_df.dropna()
sample_df["text"] = sample_df["text"].astype(str).str.strip()
sample_df = sample_df[sample_df["text"] != ""].reset_index(drop=True)
sample_df["row_id"] = sample_df.index
print(f"Sample rows: {len(sample_df)}")
sample_df.head()

## Batch chunking
Slice the sample into 15-row chunks and convert each chunk to a raw CSV string
for one LLM call apiece.

In [ ]:
chunks = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
chunk_csvs = [c[["video_id", "text"]].to_csv(index=False) for c in chunks]
print(f"{len(chunks)} chunks x ~{BATCH_SIZE} rows | first chunk chars: {len(chunk_csvs[0])}")
print(chunk_csvs[0][:600])

## LLM inference
Pass the system instruction + one CSV chunk per call to the Google AI Studio API.
Sequential calls with a 1s pause; raw responses kept in memory for the parse step.

In [ ]:
prompt_template = Path("../prompts/pass-01-grief-outrage.txt").read_text(encoding="utf-8").strip()
print(prompt_template[:400])

TRANSIENT_SLEEP = (3, 5)  # pilot; flip to (10, 10) for full run
RATE_LIMIT_SLEEP = (60, 90)

def is_rate_limited(e: Exception) -> bool:
    msg = repr(e)
    return ("429" in msg) or ("RESOURCE_EXHAUSTED" in msg)

def is_transient(e: Exception) -> bool:
    msg = repr(e)
    return any(k in msg for k in ("500", "INTERNAL", "503", "UNAVAILABLE", "HIGH DEMAND",
                                  "Timeout", "TIMEOUT", "TimeoutError", "timed out"))

def call_with_retry(full_input, tries=4):
    for attempt in range(tries):
        try:
            return client.models.generate_content(model=MODEL, contents=full_input,
                config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
                    http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS)))
        except Exception as e:
            if attempt == tries - 1:
                raise
            if is_rate_limited(e):
                sleep_time = random.uniform(*RATE_LIMIT_SLEEP)
            elif is_transient(e):
                sleep_time = random.uniform(*TRANSIENT_SLEEP)
            else:
                sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"  [Attempt {attempt+1} failed: {repr(e)}] Retrying in {sleep_time:.2f}s...")
            time.sleep(sleep_time)

chunks_queue = [sample_df.iloc[i:i + BATCH_SIZE] for i in range(0, len(sample_df), BATCH_SIZE)]
raw_responses, hard_failures = [], []
# Resume-skip: reload completed row_ids from partials when not overwriting
if not overwrite:
    try:
        done_ids = set()
        if Path(PARTIAL_RESP).exists():
            with open(PARTIAL_RESP, encoding="utf-8") as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        done_ids.update(r.get("row_ids", []))
                        raw_responses.append(r)
                    except Exception:
                        pass
        if Path(PARTIAL_FAIL).exists():
            with open(PARTIAL_FAIL, encoding="utf-8") as f:
                for line in f:
                    try:
                        e = json.loads(line)
                        if "row_id" in e:
                            done_ids.add(e["row_id"])
                            hard_failures.append(e)
                    except Exception:
                        pass
        if done_ids:
            chunks_queue = [c for c in chunks_queue if not set(c["row_id"].tolist()) <= done_ids]
            print(f"Resuming: {len(done_ids)} rows done, {len(chunks_queue)} chunks left.")
    except Exception as e:
        print(f"Resume check failed ({e!r}); starting fresh.")
else:
    for p in (PARTIAL_RESP, PARTIAL_FAIL):
        try:
            Path(p).unlink(missing_ok=True)
        except Exception:
            pass
print(f"Starting dynamic queue with {len(chunks_queue)} initial chunks...")

try:
    while chunks_queue:
        current_chunk = chunks_queue.pop(0)
        n_rows = len(current_chunk)
        csv_str = current_chunk[["video_id", "text"]].to_csv(index=False)
        try:
            resp = call_with_retry(prompt_template + "\n\nData:\n" + csv_str)
            rec = {"row_ids": current_chunk["row_id"].tolist(), "text": resp.text or ""}
            raw_responses.append(rec)
            with open(PARTIAL_RESP, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            time.sleep(1)
        except Exception as e:
            print(f"❌ Chunk failure (Size: {n_rows}): {repr(e)}")
            if is_rate_limited(e) or is_transient(e):
                print(f"   ↳ Transient exhausted after retries. Logging {n_rows} rows and skipping (no split).")
                for _, row in current_chunk.iterrows():
                    ent = {"row_id": int(row["row_id"]), "video_id": row.get("video_id"),
                           "reason": "transient_exhausted", "error": repr(e),
                           "text_length": len(str(row["text"]))}
                    hard_failures.append(ent)
                    with open(PARTIAL_FAIL, "a", encoding="utf-8") as f:
                        f.write(json.dumps({"kind": "api_failure", **ent}, ensure_ascii=False) + "\n")
                continue
            if n_rows > 1:
                print(f"   ↳ Splitting chunk of {n_rows}...")
                mid = n_rows // 2
                chunks_queue.insert(0, current_chunk.iloc[mid:])
                chunks_queue.insert(0, current_chunk.iloc[:mid])
            else:
                row = current_chunk.iloc[0]
                print(f"   ↳ Hard failure on row_id {row['row_id']}. Logging and skipping.")
                ent = {"row_id": int(row["row_id"]), "video_id": row.get("video_id"),
                       "reason": "content_failure", "error": repr(e), "text_length": len(str(row["text"]))}
                hard_failures.append(ent)
                with open(PARTIAL_FAIL, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"kind": "api_failure", **ent}, ensure_ascii=False) + "\n")
except (KeyboardInterrupt, SystemExit):
    print(f"\nInterrupted: {len(raw_responses)} chunks done, {len(chunks_queue)} left. "
          f"Partials at {PARTIAL_RESP} / {PARTIAL_FAIL}.")
    raise

print(f"\nDone. {len(raw_responses)} ok chunks, {len(hard_failures)} hard failures.")

## API probe (run once)
Single-row call to isolate model/key issues from chunk payload issues before the full loop.

In [ ]:
probe = prompt_template + "\n\n" + sample_df.head(1)[["video_id", "text"]].to_csv(index=False)
r = client.models.generate_content(
    model=MODEL,
    contents=probe,
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=2048),
)
print((r.text or "")[:1500])

## Clean & parse
Strip markdown fences from each raw response, `json.loads()` it, and map parsed
objects back to exact sample rows via the queued `row_ids`. API `hard_failures`
and unparseable chunks both flush to `LOG_PATH`.

In [ ]:
def strip_fences(raw: str) -> str:
    s = raw.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

REQUIRED_KEYS = {{"video_id", "text", "grief_or_outrage", "value_or_relationship_invoked", "family_referenced"}}

parsed_records, parse_failures = [], []
for item in raw_responses:
    row_ids, txt = item["row_ids"], item["text"]
    if not txt.strip():
        parse_failures.append({"row_ids": row_ids, "error": "empty_response"})
        continue
    try:
        objs = json.loads(strip_fences(txt))
    except Exception as e:
        parse_failures.append({"row_ids": row_ids, "error": f"json_decode: {e!r}"})
        continue
    if isinstance(objs, dict):
        objs = [objs]
    if len(objs) != len(row_ids):
        parse_failures.append({"row_ids": row_ids, "error": f"count_mismatch: {len(objs)} objs for {len(row_ids)} rows"})
    for rid, o in zip(row_ids, objs):
        missing = REQUIRED_KEYS - set(o.keys())
        if missing:
            parse_failures.append({"row_ids": [rid], "error": f"missing_keys: {sorted(missing)}"})
            continue
        o["_row_id"] = rid
        parsed_records.append(o)

with open(LOG_PATH, "w", encoding="utf-8") as f:
    for x in hard_failures:
        f.write(json.dumps({"kind": "api_failure", **x}, ensure_ascii=False) + "\n")
    for x in parse_failures:
        f.write(json.dumps({"kind": "parse_failure", **x}, ensure_ascii=False) + "\n")

print(f"parsed: {len(parsed_records)}/{len(sample_df)} rows | parse failures: {len(parse_failures)} | api failures: {len(hard_failures)}")

## Inspect & save
Write parsed records to `OUTPUT_PATH` as JSONL, reload it, and merge back to the
sample for visual QA. Merge key is `_row_id == row_id` (exact, split-safe) —
not `video_id`, which repeats across comments and would fan out the join.

In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in parsed_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Wrote {len(parsed_records)} records → {out_path}")

res_df = pd.read_json(out_path, lines=True)
merged = sample_df.merge(res_df, left_on="row_id", right_on="_row_id", how="left", suffixes=("", "_llm"))
coverage = merged["_row_id"].notna().mean()
print(f"Coverage: {coverage:.1%} ({merged['_row_id'].notna().sum()}/{len(merged)} rows matched)")
display(merged[["row_id", "video_id", "text", "grief_or_outrage", "value_or_relationship_invoked", "family_referenced"]].head(10))
print(res_df["grief_or_outrage"].astype(str).value_counts().to_string())